## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Audio as ipy_audio
import librosa
import librosa.display
from sklearn.metrics import confusion_matrix

In [ ]:
import torch

In [ ]:
from datasets import load_dataset
from datasets import Audio as hfd_audio
from transformers import pipeline
from transformers import (
    WhisperForConditionalGeneration, WhisperProcessor, AutoFeatureExtractor,
    AutoModelForAudioClassification, TrainingArguments, Trainer
)
import evaluate
from renumics import spotlight
# import gradio as gr

## Dataset

In [ ]:
dataset_id = "neerajaabhyankar/hindustani-raag-small"
hrs = load_dataset(dataset_id, revision="0dfb021e54e0e7489b90a47e23ef15f34fa740ec")
dataset_name = dataset_id.split("/")[-1]

## Explore other methods...

In [ ]:
audio_sample = hrs["train"][0]["audio"]

In [ ]:
audio_sample

## Pooling across chunks

In [ ]:
import numpy as np

def fourier_pooling(X, k=1):
    """
    Pool a sequence of embeddings via Fourier transform along the time axis.

    Args:
        X: np.ndarray of shape (seq_len, emb_dim)
        method = "lowfreq": keep the DC + first k low-frequency coefficients
        k: int, number of low-freq components to retain if method="lowfreq"

    Returns:
        pooled: np.ndarray of shape (emb_dim,)
    """
    # FFT along time axis (seq_len)
    fft_vals = np.fft.rfft(X, axis=0)   # shape: (freq_len, emb_dim)
    mag = np.abs(fft_vals)
    pooled = mag[:k+1].mean(axis=0)

    return pooled


In [ ]:
import numpy as np
import pywt

def wavelet_pooling(X, wavelet="db4", level=None, method="approx"):
    """
    Pool a sequence of embeddings via wavelet decomposition along time axis.

    Args:
        X: np.ndarray of shape (seq_len, emb_dim)
        wavelet: str, wavelet family name (e.g. 'db4', 'haar')
        level: int or None, decomposition depth (None = max possible)
        method: str, one of ["energy", "approx", "detail"]
            - "approx": use only the final approximation coeffs
            - "detail": sum over detail coeffs across all scales

    Returns:
        pooled: np.ndarray of shape (emb_dim,)
    """
    seq_len, emb_dim = X.shape
    pooled = np.zeros(emb_dim)

    for d in range(emb_dim):
        # Discrete Wavelet Transform along time for this dimension
        coeffs = pywt.wavedec(X[:, d], wavelet=wavelet, level=level)

        if method == "approx":
            # Only final low-frequency (smooth) component
            pooled[d] = coeffs[0].mean()
        elif method == "detail":
            # Aggregate all detail (high-frequency) coefficients
            pooled[d] = sum(np.abs(c).mean() for c in coeffs[1:])
        else:
            raise ValueError(f"Unknown method: {method}")

    return pooled


In [ ]:
def get_fourier_pooled_ol3_audio_embedding(audio):
    emb, ts = openl3.get_audio_embedding(
        audio["array"], audio["sampling_rate"],
        # model='music', input_repr='mel256', content_type='music',
        embedding_size=512
    )
    pooled_emb = fourier_pooling(emb)
    return pooled_emb

In [ ]:
def get_wavelet_pooled_ol3_audio_embedding(audio):
    emb, ts = openl3.get_audio_embedding(
        audio["array"], audio["sampling_rate"],
        # model='music', input_repr='mel256', content_type='music',
        embedding_size=512
    )
    pooled_emb = wavelet_pooling(emb)
    return pooled_emb

## CLAP Embeddings

In [ ]:
from transformers import ClapModel, ClapProcessor

clap_model = ClapModel.from_pretrained("laion/clap-htsat-unfused")
clap_processor = ClapProcessor.from_pretrained("laion/clap-htsat-unfused")

In [ ]:
def get_clap_audio_embedding(audio):
    if audio["sampling_rate"] != 48000:
        audio_array = librosa.resample(audio["array"], orig_sr=audio["sampling_rate"], target_sr=48000)
    else:
        audio_array = audio["array"]
    audio_input = clap_processor(audios=audio_array, return_tensors="pt", sampling_rate=48000)
    audio_embed = clap_model.get_audio_features(**audio_input)
    return audio_embed

In [ ]:
# write the embeddings
# # os.mkdir("clap-embeddings-hindustani-raag-small")
# for split in ["train", "test"]:
#     print(f"Processing {split} split")
#     for i, audio in enumerate(hrs[split]):
#         if i % 10 == 0:
#             print(f"Processing {i}th audio")
#         audio_embed = get_clap_audio_embedding(audio["audio"])
#         np.savez(f"clap-embeddings-hindustani-raag-small/{split}_{i}.npz", audio_embed=audio_embed.detach().numpy())

In [ ]:
# read the embeddings
hrs_clap_embeddings = {"train": [], "test": []}
for split in ["train", "test"]:
    print(f"Reading {split} split")
    for i in range(len(hrs[split])):
        audio_embed = np.load(f"clap-embeddings-hindustani-raag-small/{split}_{i}.npz")["audio_embed"]
        hrs_clap_embeddings[split].append(audio_embed)

In [ ]:
hrs_train = np.array(hrs_clap_embeddings["train"]).reshape(len(hrs_clap_embeddings["train"]), -1)
hrs_test = np.array(hrs_clap_embeddings["test"]).reshape(len(hrs_clap_embeddings["test"]), -1)

In [ ]:
hrs_train_labels = np.array(hrs["train"]["label"])
hrs_test_labels = np.array(hrs["test"]["label"])

## OpenL3 Embeddings

In [ ]:
import openl3

In [ ]:
limit_to_label_indices = range(5)
limit_to_label_names = hrs["train"].features["label"].names[:5]

In [ ]:
def get_mean_ol3_audio_embedding(audio):
    emb, ts = openl3.get_audio_embedding(
        audio["array"], audio["sampling_rate"],
        # model='music', input_repr='mel256', content_type='music',
        embedding_size=512
    )
    mean_emb = np.mean(emb, axis=0)
    return mean_emb

In [ ]:
# write the embeddings
emeddings_path = "ol3-embeddings-waveletpooled-hindustani-raag-small"
os.makedirs(emeddings_path, exist_ok=True)
for split in ["train", "test"]:
    print(f"Processing {split} split")
    for i, audio in enumerate(hrs[split]):
        # only process limited labels
        if audio["label"] not in limit_to_label_indices:
            continue
        if i % 10 == 0:
            print(f"Processing {i}th audio")
        if f"{split}_{i}.npz" not in os.listdir(emeddings_path):
            audio_embed = get_wavelet_pooled_ol3_audio_embedding(audio["audio"])
            np.savez(f"{emeddings_path}/{split}_{i}.npz", audio_embed=audio_embed)

In [ ]:
# # write the embeddings
# emeddings_path = "ol3-embeddings-fourierpooled-hindustani-raag-small"
# os.makedirs(emeddings_path, exist_ok=True)
# for split in ["train", "test"]:
#     print(f"Processing {split} split")
#     for i, audio in enumerate(hrs[split]):
#         # only process limited labels
#         if audio["label"] not in limit_to_label_indices:
#             continue
#         if i % 10 == 0:
#             print(f"Processing {i}th audio")
#         if f"{split}_{i}.npz" not in os.listdir(emeddings_path):
#             audio_embed = get_fourier_pooled_ol3_audio_embedding(audio["audio"])
#             np.savez(f"{emeddings_path}/{split}_{i}.npz", audio_embed=audio_embed)

In [ ]:
# read the embeddings
# emeddings_path = "ol3-embeddings-fourierpooled-hindustani-raag-small"
emeddings_path = "ol3-embeddings-waveletpooled-hindustani-raag-small"
hrs_ol3_embeddings = {"train": [], "test": []}
for split in ["train", "test"]:
    print(f"Reading {split} split")
    for i in range(len(hrs[split])):
        if f"{split}_{i}.npz" in os.listdir(emeddings_path):
            audio_embed = np.load(f"{emeddings_path}/{split}_{i}.npz")["audio_embed"]
            hrs_ol3_embeddings[split].append(audio_embed)
        else:
            # print(f"Missing {split}_{i}.npz")
            continue

In [ ]:
# len(hrs_ol3_embeddings["train"]), len(hrs_ol3_embeddings["test"])

In [ ]:
hrs_train = np.array(hrs_ol3_embeddings["train"]).reshape(len(hrs_ol3_embeddings["train"]), -1)
hrs_test = np.array(hrs_ol3_embeddings["test"]).reshape(len(hrs_ol3_embeddings["test"]), -1)

In [ ]:
hrs_train_labels = np.array(hrs["train"]["label"])
hrs_test_labels = np.array(hrs["test"]["label"])

In [ ]:
hrs_train_labels = hrs_train_labels[:len(hrs_ol3_embeddings["train"])]
hrs_test_labels = hrs_test_labels[:len(hrs_ol3_embeddings["test"])]

## UMAP of the embeddings

In [ ]:
import umap

def plot_umap(embeddings, labels, title):
    plt.figure(figsize=(10, 8))
    reducer = umap.UMAP(random_state=42)
    embedding_2d = reducer.fit_transform(embeddings)
    df = pd.DataFrame(embedding_2d, columns=["UMAP1", "UMAP2"])
    df["label"] = labels
    sns.scatterplot(data=df, x="UMAP1", y="UMAP2", hue="label", palette="tab10", s=50)
    plt.title(title)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.show()

In [ ]:
plot_umap(hrs_train, hrs_train_labels, "clap-embeddings train split")

## TSNE of the embeddings

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

def plot_tsne(embeddings, labels, title):
    tsne = TSNE(n_components=2, random_state=42)
    pca = PCA(n_components=50)
    embeddings_pca = pca.fit_transform(embeddings)
    embeddings_tsne = tsne.fit_transform(embeddings_pca)

    plt.figure(figsize=(10, 10))
    sns.scatterplot(x=embeddings_tsne[:, 0], y=embeddings_tsne[:, 1], hue=labels, palette="Set1", legend="full")
    plt.title(title)
    plt.show()

In [ ]:
plot_tsne(hrs_train, hrs_train_labels, "clap-embeddings train split")

## PCA of the Embeddings

In [ ]:
def plot_pca_2d(embeddings, labels, title):
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(embeddings)

    plt.figure(figsize=(10, 10))
    sns.scatterplot(x=embeddings_2d[:, 0], y=embeddings_2d[:, 1], hue=labels, palette="Set1", legend="full")
    plt.title(title)
    plt.show()

In [ ]:
plot_pca_2d(hrs_train, hrs_train_labels, "ol3-embeddings train split")

## Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(hrs_train, hrs_train_labels)
# make predictions
y_pred = log_reg.predict(hrs_test)
# calculate accuracy
train_accuracy = log_reg.score(hrs_train, hrs_train_labels)
test_accuracy = log_reg.score(hrs_test, hrs_test_labels)
print(f"Train Accuracy: {train_accuracy * 100:.2f}%, Test Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
# get the softmax scores
y_scores = log_reg.predict_proba(hrs_train)

In [ ]:
plt.imshow(y_scores, aspect='auto', cmap='viridis')

In [ ]:
cm = confusion_matrix(hrs_test_labels, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=limit_to_label_names, yticklabels=limit_to_label_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix for Logistic Regression on CLAP Embeddings')
plt.show()

In [ ]:
# # find the relative importance of each feature in predicting the label
# feature_importance = np.abs(log_reg.coef_).mean(axis=0)
# # get the indices of the top 10 features
# top_10_features = np.argsort(feature_importance)[-10:]
# # plot the umap using the top 10 features only
# plot_umap(hrs_train[:, top_10_features], hrs_train_labels, "Top 10 features UMAP train split")

## Train a small model on these embeddings

In [ ]:
class SmallNN(torch.nn.Module):
    def __init__(self):
        super(SmallNN, self).__init__()
        self.fc1 = torch.nn.Linear(512, 256)
        self.bn1 = torch.nn.BatchNorm1d(256)
        self.fc2 = torch.nn.Linear(256, 128)
        self.bn2 = torch.nn.BatchNorm1d(128)
        self.fc3 = torch.nn.Linear(128, 50)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.bn1(x)
        x = torch.relu(self.fc2(x))
        x = self.bn2(x)
        x = self.fc3(x)
        return x

In [ ]:
model = SmallNN()
device = torch.device("mps")
# device = torch.device("cpu")
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
epochs = 50
batch_size = 32

In [ ]:
train_dataset = torch.utils.data.TensorDataset(
    torch.tensor(hrs_train, dtype=torch.float32).to(device),
    torch.tensor(hrs_train_labels, dtype=torch.long).to(device)
)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataset = torch.utils.data.TensorDataset(
    torch.tensor(hrs_test, dtype=torch.float32).to(device),
    torch.tensor(hrs_test_labels, dtype=torch.long).to(device)
)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
def train(model, train_loader, optimizer, criterion, epochs):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        for i, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            # print(inputs.shape)
            # if smaller than batch size, pad by repeating the last element
            if inputs.shape[0] < batch_size:
                inputs = torch.cat([inputs, inputs[-1].unsqueeze(0).repeat(batch_size - inputs.shape[0], 1)])
                labels = torch.cat([labels, labels[-1].unsqueeze(0).repeat(batch_size - labels.shape[0])])
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        with torch.no_grad():
            running_val_loss = 0.0
            for i, (inputs, labels) in enumerate(test_loader):
                inputs = inputs.to(device)
                labels = labels.to(device)
                outputs = model(inputs)
                running_val_loss += criterion(outputs, labels).item()
        print(f"Epoch [{epoch+1}/{epochs}], Train Loss: {running_loss/len(train_loader):.4f}, Val Loss: {running_val_loss/len(test_loader):.4f}")
        # if (epoch + 1) % 5 == 0:
        #     torch.save(model.state_dict(), f"model_epoch_{epoch+1}.pth")

In [ ]:
train(model, train_loader, optimizer, criterion, epochs)

In [ ]:
y_pred = []
y_true = []
model.eval()
with torch.no_grad():
    for i, (inputs, labels) in enumerate(test_loader):
        inputs = inputs.to(device)
        labels = labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        y_pred.extend(predicted.cpu().numpy())
        y_true.extend(labels.cpu().numpy())


plt.scatter(y_true, y_pred)

In [ ]:
# If the model predicts the same class for all samples with probability 1
const_probabs = torch.zeros(100, 50).to(device)
const_probabs[:, 0] = 1.0
balanced_labels = torch.tensor(np.random.randint(50, size=100)).to(device)
print("constant pred loss", criterion(const_probabs, balanced_labels))